# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/GauriKale1328/flyrank-assignment/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

For my Refresh / Content Opportunity Scoring lane, I selected **Logistic Regression** as my machine learning model.

This method fits my lane because the goal is to predict whether a content page should be prioritized for refresh using search and engagement metrics. Logistic Regression is simple, easy to interpret, and suitable for binary classification. It also provides a fair comparison with the Week-4 rule-based baseline using the same features and evaluation metrics.

In [16]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

A train-test split is used to divide the dataset into training and testing sets. The same features and target definition are used for both the Week-4 baseline and the Logistic Regression model, ensuring a fair comparison.

This split provides an unbiased evaluation of the model while avoiding the use of future information during training.

In [17]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In this section, the dataset is prepared, features and target are created, and a Logistic Regression model is trained. The model is evaluated using the same data, split, and metrics as the Week-4 baseline to provide a fair comparison.

In [18]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
!pip -q install duckdb huggingface_hub pyarrow scikit-learn pandas

In [19]:
#Setup
import pandas as pd
import duckdb
from huggingface_hub import login
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")
login(token=HF_TOKEN)

con = duckdb.connect()

con.execute("""
INSTALL httpfs;
LOAD httpfs;
""")

con.execute("""
CREATE OR REPLACE SECRET hf_secret (
    TYPE HUGGINGFACE,
    TOKEN ?
)
""", [HF_TOKEN])

print("Setup completed")


Setup completed


In [20]:
### 3.1 Load the dataset
df = con.execute("""
SELECT *
FROM read_parquet(
'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/data_0.parquet'
)
LIMIT 100000
""").df()

df.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,...,sessions_ai,ai_chatgpt,ai_perplexity,ai_gemini,ai_copilot,ai_claude,ai_meta,ai_other,scroll_events,month
0,2026-03-01,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,True,False,True,<NA>,20,0,67,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
1,2026-03-01,client_73cda7b4e4f265ea,content_05597932fe4da067,True,False,True,<NA>,1,0,0,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
2,2026-03-01,client_73cda7b4e4f265ea,content_7a105f548d9c6916,True,False,True,<NA>,125,1,616,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
3,2026-03-01,client_73cda7b4e4f265ea,content_905aa32a0230694e,True,False,True,<NA>,7,0,28,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
4,2026-03-01,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,True,False,True,<NA>,11,0,25,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03


In [21]:
print(df.shape)
print(df.columns.tolist())

(100000, 31)
['report_date', 'client_hash_id', 'content_hash_id', 'client_has_gsc', 'client_has_ga4', 'gsc_data_available', 'ga4_data_available', 'gsc_impressions', 'gsc_clicks', 'gsc_sum_position', 'gsc_avg_position', 'ga4_pageviews', 'ga4_sessions', 'ga4_users', 'ga4_engaged_sessions', 'ga4_total_engagement_sec', 'sessions_organic', 'sessions_direct', 'sessions_referral', 'sessions_social', 'sessions_paid', 'sessions_ai', 'ai_chatgpt', 'ai_perplexity', 'ai_gemini', 'ai_copilot', 'ai_claude', 'ai_meta', 'ai_other', 'scroll_events', 'month']


In [22]:
### 3.2 Create features and target label
import numpy as np

# CTR calculate
df["ctr"] = np.where(
    df["gsc_impressions"] > 0,
    df["gsc_clicks"] / df["gsc_impressions"],
    0
)

# Average Position calculate
df["avg_position"] = np.where(
    df["gsc_impressions"] > 0,
    df["gsc_sum_position"] / df["gsc_impressions"],
    0
)

# Baseline score
df["baseline_score"] = 0

df.loc[df["gsc_impressions"] >= 100, "baseline_score"] += 2
df.loc[df["ctr"] < 0.02, "baseline_score"] += 2
df.loc[df["avg_position"] > 20, "baseline_score"] += 1

# Action (Target)
df["target"] = np.where(df["baseline_score"] >= 3, 1, 0)

df[["gsc_impressions", "ctr", "avg_position", "baseline_score", "target"]].head()

,gsc_impressions,ctr,avg_position,baseline_score,target
0,20,0.000,3.350000,2,0
1,1,0.000,0.000000,2,0
2,125,0.008,4.928000,4,1
3,7,0.000,4.000000,2,0
4,11,0.000,2.272727,2,0


In [23]:
### 3.3 Select features and split the data
from sklearn.model_selection import train_test_split

# Features
X = df[["gsc_impressions", "gsc_clicks", "ctr", "avg_position"]]

# Target
y = df["target"]

# Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))

Training samples: 80000
Testing samples: 20000


In [24]:
### 3.4 Train the Logistic Regression model
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(max_iter=1000)

model.fit(X_train, y_train)

print("Model trained successfully!")

Model trained successfully!


In [25]:
### 3.5 Evaluate model performance
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

y_pred = model.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

print("Accuracy :", round(accuracy, 3))
print("Precision:", round(precision, 3))
print("Recall   :", round(recall, 3))
print("F1 Score :", round(f1, 3))

Accuracy : 0.979
Precision: 0.933
Recall   : 0.927
F1 Score : 0.93


In [26]:
### 3.6 Compare the model with the Week-4 baseline
comparison = pd.DataFrame({
    "Method": ["Week 4 Rule Baseline", "Logistic Regression"],
    "Accuracy": ["Rule-based", 0.979],
    "Precision": ["Rule-based", 0.933],
    "Recall": ["Rule-based", 0.927],
    "F1 Score": ["Rule-based", 0.930]
})

comparison

,Method,Accuracy,Precision,Recall,F1 Score
0,Week 4 Rule Baseline,Rule-based,Rule-based,Rule-based,Rule-based
1,Logistic Regression,0.979,0.933,0.927,0.93


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

The Logistic Regression model performed well, but some predictions may still be incorrect. False positives may occur when pages have low CTR but do not actually need a refresh. False negatives may occur because content quality and business context are not included in the features. The model mainly relies on impressions, clicks, CTR, and average position, so it should be used as a decision-support tool rather than a final decision maker.

In [27]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.